# APS360 super-resolution: Colab runner
**Runtime → Change runtime type → T4 GPU.** Run top to bottom.

Everything important (checkpoints, logs, test sets, results) is saved to Google Drive in `aps360_sr_runs/`. If Colab disconnects or you reopen it: run cells 1–3 again (they skip work already done), then rerun the training cell — it resumes from the last checkpoint automatically.

## 1. Drive + code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/aps360_sr_runs'
!mkdir -p $RUNS
!nvidia-smi -L

import os
if not os.path.exists('/content/sr-project'):
    !git clone -q https://github.com/calikir1/progress-report.git /content/sr-project
%cd /content/sr-project
!git pull -q
!pip -q install -r requirements.txt

## 2. Data (~8 min the first time in a session)

In [ ]:
%%bash
set -e
mkdir -p data/benchmark
cd data
[ -d DIV2K_train_HR ] || (wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip && unzip -q DIV2K_train_HR.zip && rm DIV2K_train_HR.zip)
[ -d DIV2K_valid_HR ] || (wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip && unzip -q DIV2K_valid_HR.zip && rm DIV2K_valid_HR.zip)
cd benchmark
[ -d Set5 ] || (wget -q https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz && tar -xzf Set5_HR.tar.gz && mv Set5_HR Set5)
[ -d Set14 ] || (wget -q https://huggingface.co/datasets/eugenesiow/Set14/resolve/main/data/Set14_HR.tar.gz && tar -xzf Set14_HR.tar.gz && mv Set14_HR Set14)
[ -d B100 ] || (wget -q https://huggingface.co/datasets/eugenesiow/BSD100/resolve/main/data/BSD100_HR.tar.gz && tar -xzf BSD100_HR.tar.gz && mv BSD100_HR B100)
cd /content/sr-project
echo "DIV2K train $(ls data/DIV2K_train_HR | wc -l), valid $(ls data/DIV2K_valid_HR | wc -l), Set5 $(ls data/benchmark/Set5 | wc -l), Set14 $(ls data/benchmark/Set14 | wc -l), B100 $(ls data/benchmark/B100 | wc -l)"

In [ ]:
!python prepare_data.py --src data/DIV2K_train_HR --dst data/div2k_train_npy
!python prepare_data.py --src data/DIV2K_valid_HR --dst data/div2k_valid_npy

## 3. Fixed test sets
Built **once**, then always restored from Drive so every experiment uses identical files. The sha256 must end in `a0a003a`.

In [ ]:
import os
if os.path.exists(f'{RUNS}/testsets/MANIFEST.txt'):
    !rm -rf testsets && cp -r $RUNS/testsets .
    print('restored test sets from Drive')
else:
    !python make_testsets.py --bench_root data/benchmark --sets Set5 Set14 B100
    !cp -r testsets $RUNS/
!tail -1 testsets/MANIFEST.txt

## 4. One-time checks (already passed — skip after a reset)

In [ ]:
# !python smoke_test.py
# !python check_data.py --train data/DIV2K_train_HR data/DIV2K_valid_HR --bench data/benchmark/Set5 data/benchmark/Set14 data/benchmark/B100
!python evaluate.py --methods bicubic --out $RUNS/results
!python count_params.py

## 5. Train (~30–45 min each on a T4)
`--resume` continues from the last checkpoint if one exists, otherwise starts fresh. After a disconnect: rerun cells 1–3, then rerun the same training cell. Short on time? Lower `--iters` (15000 still beats bicubic).

In [ ]:
!python train.py --name clean --model edsr --iters 30000 --runs $RUNS --resume

In [ ]:
!python train.py --name degraded --model edsr --degraded --iters 30000 --runs $RUNS --resume

## 6. Results for the report

In [ ]:
!python evaluate.py --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt --out $RUNS/results
!python plot_curves.py $RUNS/clean $RUNS/degraded
!python visualize.py --set Set14 --image comic.png --condition clean --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt
!python visualize.py --set B100 --image 253027.png --condition b1.0_n5_q75 --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt
!mkdir -p $RUNS/results && cp -r results/figures $RUNS/results/

In [ ]:
from IPython.display import Image, display, Markdown
import glob
display(Markdown(open(f'{RUNS}/results/table.md').read()))
for f in sorted(glob.glob('results/figures/*.png')): print(f); display(Image(f))

## 7. Download everything for Overleaf + the repo

In [ ]:
!cd $RUNS && zip -qr /content/for_report.zip results/table.md results/summary.csv results/figures testsets/degraded_params.csv testsets/MANIFEST.txt clean/log.csv clean/config.json degraded/log.csv degraded/config.json
from google.colab import files
files.download('/content/for_report.zip')

## Later (after the progress report)

In [ ]:
# !python train.py --name degaware --model degaware --degraded --iters 30000 --runs $RUNS --resume
# !python train.py --name edsr9_control --model edsr --blocks 9 --degraded --iters 30000 --runs $RUNS --resume
# !python train.py --name srcnn --model srcnn --iters 30000 --runs $RUNS --resume
# !python evaluate.py --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt $RUNS/degaware/best.pt --out $RUNS/results
# !python visualize.py --phone_dir data/phone --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt $RUNS/degaware/best.pt